In [ ]:
from pathlib import Path

# ---- USER INPUT ----
hyp3_dir = Path("/Volumes/Fortress_L3/SnowWaterEquivalent/SWE_Shadi/HyP3/hyp3_data/clipped_data/")
pattern = "*unw_phase_clipped.tif"   # adjust if needed
wsdlurl = "https://hydroportal.cuahsi.org/Snotel/cuahsi_1_1.asmx?WSDL"

obs_hour = 0          # 0 = midnight UTC, 6 = 6AM UTC etc.
reference_date = "12-01"
include_temperature = True


In [ ]:
from utils import build_insar_context

tifs = sorted(hyp3_dir.glob(pattern))
ctx = build_insar_context(source="hyp3", hyp3_tifs=tifs)

print(ctx.source, len(ctx.dates))
ctx.dates[:5], ctx.dates[-1]


In [ ]:
from utils.snotel_utils import fetch_snotel_sites, filter_sites_by_polygon, fetch_snotel_timeseries

# 1) Fetch all SNOTEL sites (returns EPSG:4326 points)
sites = fetch_snotel_sites(wsdlurl)

# 2) Filter sites to your InSAR footprint geometry
# Assumes ctx.footprint geometry is already EPSG:4326
snotel_sites = filter_sites_by_polygon(sites, ctx.footprint.iloc[0].geometry)

print("Stations in footprint:", len(snotel_sites))

# 3) Fetch SWE + optional temperature time series for those sites
swe_data = fetch_snotel_timeseries(
    snotel_sites,
    wsdlurl,
    start_date=str(ctx.dates[0].date()),
    end_date=str(ctx.dates[-1].date()),
    reference_date="12-01",
    obs_hour=0,                # midnight UTC (change to 6 for 6AM UTC, etc.)
    include_temperature=True,
)

# Quick peek at one station
first_station = next(iter(swe_data))
swe_data[first_station].head()

In [ ]:
from utils.plotting import make_footprint_station_map

m = make_footprint_station_map(ctx.footprint, snotel_sites, zoom_start=8)
m


In [ ]:
from utils import plot_snotel_data
plot_snotel_data(swe_data, ctx.dates, x_axis="days_since_reference")

In [ ]:
from utils import save_pickle

save_pickle(swe_data, "cache/snotel_data_hyp3.pkl")
